# Week 1 — Profiling a Llama 8B model by component

This notebook loads a student-provided Hugging Face-compatible Llama 8B checkpoint and measures real model modules on one CUDA GPU. It does not download, train, or modify the model.

You will learn how to:

- distinguish asynchronous CPU launch time from GPU execution time;
- use CUDA streams and CUDA events correctly;
- profile embedding, RMSNorm, attention projections, attention, MLP projections, a decoder layer, final norm, and the LM head;
- compare latency across sequence lengths;
- export a detailed `torch.profiler` trace.


## 1. What is being measured?

```text
Token IDs
   │
   ▼
[Embedding]
   │
   ▼
[RMSNorm → Q/K/V → RoPE → Attention → O projection → residual]
   │
   ▼
[RMSNorm → Gate/Up → SwiGLU → Down projection → residual]
   │                     repeated for every decoder layer
   ▼
[Final RMSNorm → LM head → next-token logits]
```

The notebook measures one middle decoder layer because Llama decoder layers have the same structure. `attention_inclusive`, `mlp_inclusive`, and `decoder_layer_inclusive` include their child operations; do not add those rows to the child-module rows.


## 2. CUDA streams and events

A **CUDA stream** is an ordered queue of GPU work. PyTorch normally submits kernels to the current stream asynchronously, so Python can continue before the GPU finishes. Operations in one stream execute in submission order; independent operations in different streams may overlap.

```text
Python:        launch QKV ─ launch attention ─ launch MLP ─ continue
CUDA stream:  ──[ QKV kernels ]─[ attention ]─[ MLP kernels ]──▶
```

A **CUDA event** is a marker inserted into a stream. The GPU records the event when all preceding work in that stream reaches it. Two timing-enabled events therefore measure elapsed GPU time, rather than Python launch time.

```text
CUDA stream:  ──[ start event ]─[ measured kernels ]─[ end event ]──▶
                         └──── elapsed GPU time ────┘
```

We warm up first, synchronize before the experiment, record one start/end pair for each repetition, and synchronize once after all repetitions. Other GPU workloads can still contend for resources and change the measurement.


In [ ]:
from __future__ import annotations

import gc
import os
import platform
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import AutoConfig, AutoModelForCausalLM

print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"CUDA ready:   {torch.cuda.is_available()}")


## 3. Student configuration

Set `LLAMA_MODEL_PATH` before starting Jupyter or enter your own local checkpoint directory when prompted. No personal path is stored in this notebook. The default experiment uses batch size 1 and sequence lengths 128, 512, and 1024 tokens.


In [ ]:
model_path_text = os.environ.get("LLAMA_MODEL_PATH", "").strip()
if not model_path_text:
    model_path_text = input("Local Hugging Face Llama 8B directory: " ).strip()
if not model_path_text:
    raise ValueError("A local model directory is required.")

MODEL_PATH = Path(model_path_text).expanduser().resolve()
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model directory does not exist: {MODEL_PATH}")

DEVICE = torch.device(os.environ.get("PROFILE_DEVICE", "cuda:0"))
DTYPE_NAME = os.environ.get("PROFILE_DTYPE", "bfloat16").lower()
ATTN_IMPLEMENTATION = os.environ.get("LLAMA_ATTN_IMPLEMENTATION", "sdpa")
BATCH_SIZE = int(os.environ.get("PROFILE_BATCH_SIZE", "1"))
SEQUENCE_LENGTHS = [
    int(value)
    for value in os.environ.get("PROFILE_SEQUENCE_LENGTHS", "128,512,1024").split(",")
]
WARMUP_STEPS = int(os.environ.get("PROFILE_WARMUP_STEPS", "3"))
REPEAT_STEPS = int(os.environ.get("PROFILE_REPEAT_STEPS", "10"))
OUTPUT_DIR = Path(os.environ.get("PROFILE_OUTPUT_DIR", "profile-results")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE.type != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("This profiling exercise requires a CUDA GPU.")
if min(SEQUENCE_LENGTHS) <= 0 or BATCH_SIZE <= 0:
    raise ValueError("Batch size and sequence lengths must be positive.")

print(f"Model directory: {MODEL_PATH}")
print(f"Device:          {DEVICE}")
print(f"Dtype:           {DTYPE_NAME}")
print(f"Attention:       {ATTN_IMPLEMENTATION}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Lengths:         {SEQUENCE_LENGTHS}")
print(f"Output:          {OUTPUT_DIR}")


## 4. Load the actual model

Weights are loaded only from the directory supplied above. The notebook checks the architecture and reports the model dimensions used by later measurements.


In [ ]:
dtype_by_name = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
if DTYPE_NAME not in dtype_by_name:
    raise ValueError(f"Unsupported PROFILE_DTYPE={DTYPE_NAME!r}; choose {sorted(dtype_by_name)}")
DTYPE = dtype_by_name[DTYPE_NAME]
if DTYPE is torch.bfloat16 and not torch.cuda.is_bf16_supported():
    raise RuntimeError("The selected GPU does not support bfloat16; set PROFILE_DTYPE=float16.")

config = AutoConfig.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)
if config.model_type != "llama":
    raise ValueError(f"Expected a Llama checkpoint, found model_type={config.model_type!r}.")

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    attn_implementation=ATTN_IMPLEMENTATION,
).to(DEVICE).eval()
model.config.use_cache = False
torch.cuda.synchronize(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
summary = {
    "parameters": parameter_count,
    "layers": config.num_hidden_layers,
    "hidden_size": config.hidden_size,
    "intermediate_size": config.intermediate_size,
    "attention_heads": config.num_attention_heads,
    "kv_heads": config.num_key_value_heads,
    "head_dim": head_dim,
    "vocabulary_size": config.vocab_size,
}
display(pd.Series(summary, name="value").to_frame())
print(f"Loaded parameters: {parameter_count / 1e9:.3f}B")
print(f"Allocated GPU memory: {torch.cuda.memory_allocated(DEVICE) / 2**30:.2f} GiB")


## 5. Select and capture model components

A forward pre-hook records the real arguments received by each selected module during one full model pass. The hooks are then removed. Later timing replays those exact calls in isolation.


In [ ]:
base_model = model.model
layer_index = len(base_model.layers) // 2
layer = base_model.layers[layer_index]
attention = layer.self_attn
mlp = layer.mlp

modules: OrderedDict[str, torch.nn.Module] = OrderedDict([
    ("embedding", base_model.embed_tokens),
    ("decoder_layer_inclusive", layer),
    ("input_rmsnorm", layer.input_layernorm),
    ("attention_inclusive", attention),
    ("q_proj", attention.q_proj),
    ("k_proj", attention.k_proj),
    ("v_proj", attention.v_proj),
    ("o_proj", attention.o_proj),
    ("post_attention_rmsnorm", layer.post_attention_layernorm),
    ("mlp_inclusive", mlp),
    ("gate_proj", mlp.gate_proj),
    ("up_proj", mlp.up_proj),
    ("down_proj", mlp.down_proj),
    ("final_rmsnorm", base_model.norm),
    ("lm_head_last_token", model.lm_head),
])

rotary_module = getattr(base_model, "rotary_emb", getattr(attention, "rotary_emb", None))
if isinstance(rotary_module, torch.nn.Module):
    modules["rotary_preparation"] = rotary_module
if isinstance(getattr(mlp, "act_fn", None), torch.nn.Module):
    modules["activation"] = mlp.act_fn

INCLUSIVE_COMPONENTS = {
    "attention_inclusive",
    "mlp_inclusive",
    "decoder_layer_inclusive",
}

@dataclass
class CapturedCall:
    module: torch.nn.Module
    args: tuple[Any, ...]
    kwargs: dict[str, Any]

def capture_calls(input_ids: torch.Tensor, attention_mask: torch.Tensor) -> OrderedDict[str, CapturedCall]:
    captured: OrderedDict[str, CapturedCall] = OrderedDict()
    handles = []

    def make_hook(name: str):
        def hook(module: torch.nn.Module, args: tuple[Any, ...], kwargs: dict[str, Any]):
            if name not in captured:
                captured[name] = CapturedCall(module=module, args=args, kwargs=dict(kwargs))
        return hook

    for name, module in modules.items():
        handles.append(module.register_forward_pre_hook(make_hook(name), with_kwargs=True))

    try:
        with torch.inference_mode():
            hidden = base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            ).last_hidden_state
            _ = model.lm_head(hidden[:, -1:, :])
        torch.cuda.synchronize(DEVICE)
    finally:
        for handle in handles:
            handle.remove()

    missing = [name for name in modules if name not in captured]
    if missing:
        print(f"Not called by this model path: {missing}")
    return captured

print(f"Profiling decoder layer {layer_index} of {len(base_model.layers)}")
print("Selected modules:", ", ".join(modules))


## 6. CUDA-event benchmark

The timer below records event pairs in the current stream. It does not place a synchronization between individual repetitions; all event timestamps are read after one final synchronization.


In [ ]:
def invoke(captured: CapturedCall):
    return captured.module(*captured.args, **captured.kwargs)

def benchmark_cuda_events(
    fn: Callable[[], Any],
    warmup_steps: int,
    repeat_steps: int,
) -> dict[str, float]:
    with torch.inference_mode():
        for _ in range(warmup_steps):
            output = fn()
            del output
    torch.cuda.synchronize(DEVICE)

    baseline_bytes = torch.cuda.memory_allocated(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    event_pairs = []

    with torch.inference_mode():
        for _ in range(repeat_steps):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record(torch.cuda.current_stream(DEVICE))
            output = fn()
            end.record(torch.cuda.current_stream(DEVICE))
            event_pairs.append((start, end))
            del output

    torch.cuda.synchronize(DEVICE)
    samples_ms = np.asarray([start.elapsed_time(end) for start, end in event_pairs], dtype=np.float64)
    peak_extra_bytes = max(0, torch.cuda.max_memory_allocated(DEVICE) - baseline_bytes)
    return {
        "mean_ms": float(samples_ms.mean()),
        "median_ms": float(np.median(samples_ms)),
        "p90_ms": float(np.percentile(samples_ms, 90)),
        "minimum_ms": float(samples_ms.min()),
        "maximum_ms": float(samples_ms.max()),
        "std_ms": float(samples_ms.std(ddof=1)) if len(samples_ms) > 1 else 0.0,
        "peak_extra_mib": peak_extra_bytes / 2**20,
        "samples": len(samples_ms),
    }


In [ ]:
torch.manual_seed(0)
rows = []

for sequence_length in SEQUENCE_LENGTHS:
    print(f"Capturing and profiling L={sequence_length} ...")
    input_ids = torch.randint(
        low=0,
        high=config.vocab_size,
        size=(BATCH_SIZE, sequence_length),
        device=DEVICE,
        dtype=torch.long,
    )
    attention_mask = torch.ones_like(input_ids)
    captured_calls = capture_calls(input_ids, attention_mask)

    for component, captured in captured_calls.items():
        stats = benchmark_cuda_events(
            lambda captured=captured: invoke(captured),
            warmup_steps=WARMUP_STEPS,
            repeat_steps=REPEAT_STEPS,
        )
        rows.append({
            "sequence_length": sequence_length,
            "batch_size": BATCH_SIZE,
            "component": component,
            "scope": "inclusive" if component in INCLUSIVE_COMPONENTS else "leaf_or_standalone",
            "dtype": DTYPE_NAME,
            "attention_backend": ATTN_IMPLEMENTATION,
            **stats,
        })

    del captured_calls, input_ids, attention_mask
    gc.collect()
    torch.cuda.empty_cache()

results = pd.DataFrame(rows).sort_values(["sequence_length", "scope", "median_ms"])
csv_path = OUTPUT_DIR / "llama8b_component_latency.csv"
results.to_csv(csv_path, index=False)
display(results)
print(f"Saved {csv_path}")


## 7. Visualize component latency

The first figure compares non-inclusive components at the largest tested sequence length. The second shows how the inclusive attention, MLP, and decoder-layer measurements scale with sequence length.


In [ ]:
largest_length = max(SEQUENCE_LENGTHS)
leaf_data = (
    results[
        (results.sequence_length == largest_length)
        & (results.scope == "leaf_or_standalone")
    ]
    .sort_values("median_ms")
)

fig, ax = plt.subplots(figsize=(9, max(4, 0.42 * len(leaf_data))))
ax.barh(leaf_data.component, leaf_data.median_ms)
ax.set_xlabel("Median GPU latency (ms)")
ax.set_ylabel("Component")
ax.set_title(f"Llama component latency, B={BATCH_SIZE}, L={largest_length}")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
component_figure = OUTPUT_DIR / "llama8b_component_latency.png"
fig.savefig(component_figure, dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for component in ["attention_inclusive", "mlp_inclusive", "decoder_layer_inclusive"]:
    series = results[results.component == component].sort_values("sequence_length")
    if not series.empty:
        ax.plot(series.sequence_length, series.median_ms, marker="o", label=component)
ax.set_xscale("log", base=2)
ax.set_xlabel("Sequence length (tokens)")
ax.set_ylabel("Median GPU latency (ms)")
ax.set_title(f"Inclusive latency scaling, B={BATCH_SIZE}")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
scaling_figure = OUTPUT_DIR / "llama8b_latency_scaling.png"
fig.savefig(scaling_figure, dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved {component_figure}")
print(f"Saved {scaling_figure}")


## 8. Detailed operator and kernel trace

`torch.profiler` records CPU operators, CUDA kernels, tensor shapes, memory activity, and supported FLOP estimates. This is more intrusive than CUDA-event timing, so use it for attribution rather than the primary latency number. The trace below covers one complete middle decoder layer at the largest configured sequence length.


In [ ]:
trace_length = max(SEQUENCE_LENGTHS)
trace_input_ids = torch.randint(
    0, config.vocab_size, (BATCH_SIZE, trace_length), device=DEVICE, dtype=torch.long
)
trace_attention_mask = torch.ones_like(trace_input_ids)
trace_calls = capture_calls(trace_input_ids, trace_attention_mask)
selected_layer_call = trace_calls["decoder_layer_inclusive"]

with torch.inference_mode():
    for _ in range(WARMUP_STEPS):
        output = invoke(selected_layer_call)
        del output
torch.cuda.synchronize(DEVICE)

activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
with torch.profiler.profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_flops=True,
) as profiler:
    with torch.profiler.record_function("selected_llama_decoder_layer"):
        with torch.inference_mode():
            output = invoke(selected_layer_call)
            del output
    torch.cuda.synchronize(DEVICE)

print(profiler.key_averages().table(sort_by="self_cuda_time_total", row_limit=25))
trace_path = OUTPUT_DIR / "llama8b_decoder_layer_trace.json"
profiler.export_chrome_trace(str(trace_path))
print(f"Saved {trace_path}")

del trace_calls, trace_input_ids, trace_attention_mask, selected_layer_call
gc.collect()
torch.cuda.empty_cache()


## 9. Questions to answer

1. Which projection has the highest median latency, and does that match its matrix dimensions?
2. Which components scale approximately linearly as sequence length grows?
3. Does inclusive attention grow faster than the MLP over the tested lengths?
4. How large is the gap between median and P90 latency? Was the GPU otherwise idle?
5. Which CUDA kernels dominate the detailed decoder-layer trace?
6. Repeat with another batch size or attention backend, changing only one variable at a time.

Do not interpret the sum of isolated component measurements as exact end-to-end model latency. Kernel fusion, allocator state, launch ordering, cache state, and framework overhead differ between isolated replay and a complete model forward pass.
